# Lab 4: Tool calling and structured output

Request a JSON response that your application can validate, then let the model request a function and use its result.

**Prerequisites:** [setup](../docs/00_prework.md), a working team key, and familiarity with a basic chat request. Allow about 25 minutes.

Structured output controls the response format. Tool calling produces a request for your application to execute a function; the model does not run the function itself.

[Notebook index](README.md) · [API reference](../docs/inference.md) · [Troubleshooting](../docs/troubleshooting.md#inference)

These examples use the workshop service configuration. Confirm aliases and quotas with the organizers. Event details used as sample data are fictional.


## Step 1: Set up your environment

In [ ]:
# Install the packages used in this notebook.
%pip install -q "openai>=3,<4" python-dotenv pydantic

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Run Jupyter from the workshop directory or its notebooks directory.
workshop_dir = Path.cwd().resolve()
if workshop_dir.name == "notebooks":
    workshop_dir = workshop_dir.parent
env_file = workshop_dir / ".env"
if not (workshop_dir / ".env.example").is_file() or not env_file.is_file():
    raise RuntimeError("Open Jupyter from the workshop directory after completing docs/00_prework.md.")
load_dotenv(env_file, override=True)

base_url = os.environ.get("OPENAI_BASE_URL", "").strip().rstrip("/")
api_key = os.environ.get("OPENAI_API_KEY", "").strip()
if not base_url or not api_key or api_key == "hk_paste_your_team_key_here":
    raise RuntimeError("Set OPENAI_BASE_URL and your team key in the workshop .env file.")
print("Endpoint:", base_url)
print("Team key loaded. Key value is hidden.")
client = OpenAI(base_url=base_url, api_key=api_key, timeout=60.0)


## Request reference

Both features use `POST /v1/chat/completions`.

| Field | Purpose |
|---|---|
| `response_format` | Request JSON that follows a supplied schema |
| `tools` | Describe the functions available to the model |
| `tool_choice` | Let the model select a function or require a named function |

Tool requests appear in `message.tool_calls`. Parse the JSON arguments, validate them, and dispatch only functions your application permits.

In [ ]:
import json

response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Give me a color and its hex code."}],
    response_format={"type": "json_schema", "json_schema": {"name": "color", "schema": {
        "type": "object",
        "properties": {"name": {"type": "string"}, "hex": {"type": "string"}},
        "required": ["name", "hex"], "additionalProperties": False}}},
    max_tokens=60,
)
if response.choices[0].finish_reason != "stop" or not response.choices[0].message.content:
    raise RuntimeError("No complete structured response. Check finish_reason and output budget.")
print(json.loads(response.choices[0].message.content))

## Step 2: Ask for JSON in the prompt

A prompt alone may return JSON, surrounding prose, or another shape. Run the request and inspect whether `json.loads` accepts it. The sample event and date are fictional.

In [ ]:
import json

response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Extract the event details as JSON with keys name, city, and date: "
               "Example Build Weekend runs in Kansas City on October 11, 2026."}],
    max_tokens=120,
)
text = response.choices[0].message.content
print(text)

try:
    print("\nParsed:", json.loads(text))
except json.JSONDecodeError as e:
    print("\njson.loads failed:", e)

## Step 3: Request a schema

Pass a schema with `response_format`. Check for a complete response before parsing it, then validate the result against your application's requirements. A token limit or service error can still prevent a usable result.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "city": {"type": "string"},
        "date": {"type": "string", "description": "ISO 8601 date"},
    },
    "required": ["name", "city", "date"],
    "additionalProperties": False,
}

response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Extract the event details: Example Build Weekend runs in Kansas City on October 11, 2026."}],
    response_format={"type": "json_schema", "json_schema": {"name": "event", "schema": schema}},
    max_tokens=120,
)
if response.choices[0].finish_reason != "stop" or not response.choices[0].message.content:
    raise RuntimeError("No complete structured response. Check finish_reason and output budget.")
text = response.choices[0].message.content
print(text)
event = json.loads(text)  # Parse the completed response.
print("\nCity:", event["city"])

A Pydantic model can generate the schema and validate the response. The next example uses one model definition for both tasks.

In [ ]:
from pydantic import BaseModel, Field

class Idea(BaseModel):
    title: str = Field(description="Short project name")
    pitch: str = Field(description="One sentence")
    difficulty: int = Field(ge=1, le=5)
    apis_used: list[str]

class Ideas(BaseModel):
    ideas: list[Idea]

response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Suggest two hackathon project ideas that use a chat model and an image model."}],
    response_format={"type": "json_schema", "json_schema": {"name": "ideas", "schema": Ideas.model_json_schema()}},
    max_tokens=1000,  # Two ideas took 195 to 396 tokens in tests; leave headroom so finish_reason is stop.
)
if response.choices[0].finish_reason != "stop" or not response.choices[0].message.content:
    raise RuntimeError("No complete structured response. Check finish_reason and output budget.")
ideas = Ideas.model_validate_json(response.choices[0].message.content)
for idea in ideas.ideas:
    print(f"{idea.title} (difficulty {idea.difficulty}): {idea.pitch}")

Schema validation checks structure and types. It does not establish that a date, amount, or other generated value is true.

## Step 4: Define an application function

Describe the function's name, purpose, and parameters. The model can return a `tool_calls` entry. Your code decides whether the call is allowed, runs it, and returns a `tool` message linked by `tool_call_id`.

The example uses a read-only lookup over fictional schedule data.

In [ ]:
# Fake data so the lab has no external dependencies. Replace with real lookups in your project.
SCHEDULE = {
    "opening": "Saturday 9:00 AM, main hall",
    "submissions": "Sunday 12:00 PM, via Devpost",
    "judging": "Sunday 1:00 PM to 3:30 PM",
    "closing": "Sunday 4:00 PM, main hall",
}

def get_schedule(event: str) -> str:
    return SCHEDULE.get(event.lower(), f"No schedule entry for '{event}'")

tools = [{
    "type": "function",
    "function": {
        "name": "get_schedule",
        "description": "Look up when a hackathon event happens. Events: opening, submissions, judging, closing.",
        "parameters": {
            "type": "object",
            "properties": {"event": {"type": "string", "description": "Event name"}},
            "required": ["event"],
        },
    },
}]

messages = [{"role": "user", "content": "When is judging?"}]
response = client.chat.completions.create(model="hackathon-chat", messages=messages, tools=tools, max_tokens=200)

message = response.choices[0].message
print("Finish reason:", response.choices[0].finish_reason)
print("Content:     ", repr(message.content))
print("Tool calls:  ", message.tool_calls)

Inspect `message.tool_calls` before dispatch. A model may return no call, one call, or several. The next cell validates the function name and arguments and returns a result for every requested call.

In [ ]:
def dispatch(call):
    if call.function.name != "get_schedule":
        raise ValueError(f"Unsupported function: {call.function.name}")
    args = json.loads(call.function.arguments)
    if not isinstance(args, dict) or set(args) != {"event"} or not isinstance(args["event"], str):
        raise ValueError("Expected one string argument named event.")
    return get_schedule(**args)

if not message.tool_calls:
    raise RuntimeError("The model did not request a tool. Try again or use Step 6 to require one.")
messages.append(message)
for call in message.tool_calls:
    result = dispatch(call)
    print(f"Ran {call.function.name}: {result}")
    messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

final = client.chat.completions.create(model="hackathon-chat", messages=messages, max_tokens=200)
print("Answer:", final.choices[0].message.content)


## Step 5: Run a bounded tool loop

Call the model, validate and run requested functions, append their results, and repeat until the model returns an answer. Bound the number of rounds to prevent an unending loop. Keep results concise because they become part of the next model request.

In [ ]:
def run(question, max_steps=5):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        response = client.chat.completions.create(model="hackathon-chat", messages=messages, tools=tools, max_tokens=300)
        message = response.choices[0].message
        messages.append(message)
        if not message.tool_calls:
            return message.content
        for call in message.tool_calls:  # The model may request several at once.
            result = dispatch(call)
            print(f"  step {step + 1}: {call.function.name} -> {result!r}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return "Stopped after too many steps."

print(run("When do submissions close, and when is the closing ceremony?"))

Inspect the function calls and combined answer. The number and grouping of calls can vary between runs. Check that the answer follows the function results.

## Step 6: Control tool use

Use the default automatic selection for conversational tasks or require a named function when the workflow needs it. To disable tools on this workshop deployment, omit `tools` from the request. The supplied materials report unreliable behavior with `tool_choice="none"`, so this lab avoids it.

In [ ]:
# Force the call even though the question does not obviously need it.
forced = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Tell me about the opening."}],
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "get_schedule"}},
    max_tokens=100,
)
print("Forced:", forced.choices[0].message.tool_calls[0].function.arguments)

# No tools in the request: the model answers from what it knows, which here is nothing useful.
plain = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "When is judging at this hackathon? One sentence."}],
    max_tokens=60,
)
print("No tools:", plain.choices[0].message.content)

## Use this in an application

Validate arguments and authorize actions before execution. Use an explicit function allow list; do not evaluate arbitrary model output as code. Require user confirmation for actions your application treats as consequential.

Use tools to collect data, then make a final schema-constrained call if your frontend needs structured fields. The starter's [tools.py](../agent/tools.py) and [code guide](../agent/README.md#add-a-tool) show where to add a function.

## Validation

In [ ]:
check = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Return the number forty-two."}],
    response_format={"type": "json_schema", "json_schema": {"name": "n", "schema": {
        "type": "object", "properties": {"value": {"type": "integer"}}, "required": ["value"], "additionalProperties": False}}},
    max_tokens=20,
)
if check.choices[0].finish_reason != "stop" or not check.choices[0].message.content:
    raise RuntimeError("No complete structured response. Check finish_reason and output budget.")
assert json.loads(check.choices[0].message.content)["value"] == 42
assert message.tool_calls and message.tool_calls[0].function.name == "get_schedule"
print("Lab 4 complete.")

## Clean up and continue

No cloud resources were created. Clear notebook outputs before sharing.

Deploy the application code with the [Linode lab](../docs/03_body.md) or [LKE lab](../docs/04_lke.md). Use [PostgreSQL](../docs/02_memory.md) for records that must persist across restarts. Return to [Build your project](../docs/build.md) to prepare a demonstration.